# Scaffolding Permit Compliance Map — Raster Edition

Uses ray-accumulated scaffolding relevance rasters (from 1M+ Cyclomedia
embeddings) instead of individual rerank detections. Advantages:

- **Spatial accuracy** — rays project detections from camera to scaffold location
- **Aggregation** — converging rays from multiple viewpoints triangulate true position
- **Type classification** — green/white signed-diff raster encodes scaffold type
- **Continuous confidence** — raster values aggregate many observations

**Pipeline:**  
GeoTIFF rasters → hotspot extraction (scipy connected components) → DoB permit
cross-reference → compliance classification (permitted / expired / unpermitted)

| Status | Color | Meaning |
|--------|-------|---------|
| **Permitted** | Green | Active DoB scaffold/shed filing within 100m |
| **Expired** | Orange | DoB filing exists but permit expired or work signed off |
| **Unpermitted** | Red | No DoB scaffold/shed filing within 100m |

**Scaffold type** (from signed\_diff raster):  
Green circle outline = standard green scaffolding, Purple = white/arched (Urban Umbrella)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import json
import requests
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import folium
from folium.plugins import HeatMap
from folium.raster_layers import ImageOverlay
from scipy import ndimage
from scipy.spatial import cKDTree
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Raster paths ---
GREEN_DIR = Path('/share/pierson/matt/mllmsci/outputs/raster/green_scaffolding_20260408_175952')
WHITE_DIR = Path('/share/pierson/matt/mllmsci/outputs/raster/white_scaffolding_20260408_175952')
COMBINE_DIR = Path('/share/pierson/matt/mllmsci/outputs/raster/scaffolding_combined_20260408_175952')

# --- Cache / output ---
CACHE_DIR = Path('/share/pierson/matt/mllmsci/notebooks/scaffolding/cache')
OUTPUT_DIR = Path('/share/pierson/matt/mllmsci/notebooks/scaffolding')
CACHE_DIR.mkdir(exist_ok=True)

# --- DoB API ---
DOB_FILINGS_URL = 'https://data.cityofnewyork.us/resource/w9ak-ipjd.json'
DOB_PERMITS_URL = 'https://data.cityofnewyork.us/resource/ipu4-2q9a.json'

# --- Detection parameters ---
DETECT_PERCENTILE = 95    # percentile threshold for hotspot detection
MIN_AREA_PIXELS = 3       # minimum hotspot size (pixels; 1 pixel = 100m2 at 10m res)
MATCH_RADIUS_M = 100      # spatial match radius for DoB permits

NYC_SP = 'EPSG:2263'
WGS84 = 'EPSG:4326'
TODAY = pd.Timestamp.now(tz='UTC')

print(f'Analysis date:      {TODAY.strftime("%Y-%m-%d")}')
print(f'Detection threshold: p{DETECT_PERCENTILE}')
print(f'Min hotspot area:   {MIN_AREA_PIXELS} pixels ({MIN_AREA_PIXELS * 100}m\u00b2)')
print(f'Match radius:       {MATCH_RADIUS_M}m')

## 1. Load Scaffolding Rasters

GeoTIFFs from the `artifact_gen` dagspace. Each pixel value is the min-max
normalized ray-accumulated cosine similarity for that query, at 10m resolution.
1,038,932 Cyclomedia face images contribute via directional ray casting.

In [ ]:
# Load all rasters
src_green = rasterio.open(next(GREEN_DIR.glob('*.tif')))
src_white = rasterio.open(next(WHITE_DIR.glob('*.tif')))
src_signed = rasterio.open(COMBINE_DIR / 'scaffolding_types_signed_diff.tif')
src_abs = rasterio.open(COMBINE_DIR / 'scaffolding_types_abs_diff.tif')

green = src_green.read(1)
white = src_white.read(1)
signed_diff = src_signed.read(1)
abs_diff = src_abs.read(1)

# Load metadata
with open(next(GREEN_DIR.glob('*_metadata.json'))) as f:
    meta_green = json.load(f)
with open(next(WHITE_DIR.glob('*_metadata.json'))) as f:
    meta_white = json.load(f)

bounds = src_green.bounds
extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
resolution_m = meta_green['resolution_m']

# Type-agnostic detection: max of green and white at each pixel
combined = np.fmax(green, white)
valid = combined[~np.isnan(combined)]

print(f'Grid: {green.shape[1]} x {green.shape[0]} @ {resolution_m}m')
print(f'BBox: {meta_green["bbox"]}')
print(f'Input points: {meta_green["n_input_points"]:,}')
print(f'Valid pixels: {len(valid):,} / {combined.size:,} ({100*len(valid)/combined.size:.1f}%)')
print(f'\nGreen stats:    mean={np.nanmean(green):.4f}, p95={np.nanpercentile(green[~np.isnan(green)], 95):.4f}')
print(f'White stats:    mean={np.nanmean(white):.4f}, p95={np.nanpercentile(white[~np.isnan(white)], 95):.4f}')
print(f'Combined stats: mean={valid.mean():.4f}, p95={np.percentile(valid, 95):.4f}')

In [ ]:
# Quick visual overview
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, raster, title, cmap in [
    (axes[0], green, 'Green Scaffolding', 'Greens'),
    (axes[1], white, 'White/Arched Scaffolding', 'Purples'),
    (axes[2], combined, 'Combined max(G, W)', 'inferno'),
]:
    im = ax.imshow(raster, extent=extent, cmap=cmap, vmin=0, vmax=0.5,
                   origin='upper', interpolation='nearest')
    ax.set_xlabel('Longitude')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Relevance', shrink=0.5)

axes[0].set_ylabel('Latitude')
plt.suptitle('Scaffolding Relevance Rasters (ray accumulation, 10m res)', y=0.95)
plt.tight_layout()
plt.show()

## 2. Extract Scaffold Hotspots

Threshold `max(green, white)` at p95 to find high-relevance regions, then extract
connected components as discrete scaffold locations. Each hotspot centroid
represents the estimated physical scaffold position (triangulated by converging rays).

Scaffold **type** (green vs. white) is determined from the `signed_diff` raster at
each hotspot centroid.

In [ ]:
# Threshold and label connected components
threshold = np.percentile(valid, DETECT_PERCENTILE)
binary = (~np.isnan(combined)) & (combined >= threshold)
labeled, n_total = ndimage.label(binary)

print(f'Threshold (p{DETECT_PERCENTILE}): {threshold:.4f}')
print(f'Total connected regions: {n_total:,}')

# Compute stats for each hotspot
hotspots = []
for label_id in range(1, n_total + 1):
    mask = labeled == label_id
    n_pix = int(mask.sum())
    if n_pix < MIN_AREA_PIXELS:
        continue

    region_vals = combined[mask]
    rows, cols = np.where(mask)

    # Centroid in geographic coordinates
    center_col = cols.mean()
    center_row = rows.mean()
    lon = bounds.left + (center_col + 0.5) * src_green.res[0]
    lat = bounds.top - (center_row + 0.5) * abs(src_green.res[1])

    # Scaffold type from signed_diff at centroid pixel
    cr, cc = int(round(center_row)), int(round(center_col))
    cr = np.clip(cr, 0, signed_diff.shape[0] - 1)
    cc = np.clip(cc, 0, signed_diff.shape[1] - 1)
    sd_val = signed_diff[cr, cc]

    if np.isfinite(sd_val):
        scaffold_type = 'green' if sd_val > 0.005 else ('white' if sd_val < -0.005 else 'ambiguous')
    else:
        scaffold_type = 'ambiguous'

    hotspots.append({
        'lat': lat,
        'lon': lon,
        'n_pixels': n_pix,
        'area_m2': n_pix * resolution_m ** 2,
        'max_value': float(region_vals.max()),
        'mean_value': float(region_vals.mean()),
        'signed_diff': float(sd_val) if np.isfinite(sd_val) else 0.0,
        'scaffold_type': scaffold_type,
    })

hotspots_df = pd.DataFrame(hotspots).sort_values('max_value', ascending=False).reset_index(drop=True)

print(f'\nHotspots after area filter (>= {MIN_AREA_PIXELS} pixels): {len(hotspots_df):,}')
print(f'\nScaffold type distribution:')
print(hotspots_df.scaffold_type.value_counts().to_string())
print(f'\nArea (m\u00b2): mean={hotspots_df.area_m2.mean():.0f}, '
      f'median={hotspots_df.area_m2.median():.0f}, max={hotspots_df.area_m2.max():.0f}')
print(f'Max relevance: mean={hotspots_df.max_value.mean():.3f}, '
      f'top-10 mean={hotspots_df.head(10).max_value.mean():.3f}')

In [ ]:
# Visualize hotspots on the raster
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: hotspot map
ax = axes[0]
im = ax.imshow(combined, extent=extent, cmap='inferno', vmin=0, vmax=0.5,
               origin='upper', interpolation='nearest', alpha=0.7)

type_colors = {'green': '#2ecc71', 'white': '#9b59b6', 'ambiguous': '#95a5a6'}
for t, color in type_colors.items():
    subset = hotspots_df[hotspots_df.scaffold_type == t]
    ax.scatter(subset.lon, subset.lat, s=subset.area_m2.clip(upper=2000) / 30,
              c=color, alpha=0.6, edgecolors='white', linewidths=0.3,
              label=f'{t} ({len(subset)})')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Scaffold Hotspots (n={len(hotspots_df):,})')
ax.legend(fontsize=9, loc='lower right')
plt.colorbar(im, ax=ax, label='Relevance', shrink=0.5)

# Right: area vs max value scatter
ax = axes[1]
for t, color in type_colors.items():
    subset = hotspots_df[hotspots_df.scaffold_type == t]
    ax.scatter(subset.area_m2, subset.max_value, s=10, c=color,
              alpha=0.5, label=f'{t} ({len(subset)})')
ax.set_xlabel('Hotspot Area (m\u00b2)')
ax.set_ylabel('Max Relevance')
ax.set_title('Hotspot Size vs. Confidence')
ax.set_xscale('log')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Fetch DoB Permit Data

Same as the rerank-based notebook — reuses cached data if available.
Dataset 1 (filings) for scaffold/shed flags + `first_permit_date` / `signoff_date`.
Dataset 2 (permit issuance) via BIN lookup for `expiration_date`.

In [ ]:
def fetch_socrata(url, where, select, limit=50000, cache_key=None):
    """Paginated Socrata SODA API fetch with local parquet cache."""
    if cache_key:
        cache_path = CACHE_DIR / f'{cache_key}.parquet'
        if cache_path.exists():
            df = pd.read_parquet(cache_path)
            print(f'  Loaded {len(df):,} rows from cache ({cache_path.name})')
            return df

    records = []
    offset = 0
    while True:
        params = {
            '$where': where,
            '$select': select,
            '$limit': limit,
            '$offset': offset,
            '$order': ':id',
        }
        r = requests.get(url, params=params, timeout=120)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        records.extend(batch)
        print(f'  Fetched {len(records):,} records...', end='\r')
        offset += limit
        if len(batch) < limit:
            break

    df = pd.DataFrame(records)
    print(f'  Fetched {len(records):,} records total.    ')
    if cache_key and len(df) > 0:
        df.to_parquet(CACHE_DIR / f'{cache_key}.parquet', index=False)
        print(f'  Cached to {cache_key}.parquet')
    return df

In [ ]:
FILINGS_COLS = ','.join([
    'job_filing_number', 'filing_status', 'filing_date',
    'first_permit_date', 'current_status_date', 'signoff_date',
    'latitude', 'longitude', 'scaffold', 'shed',
    'borough', 'house_no', 'street_name', 'block', 'lot', 'bin',
    'initial_cost', 'job_type'
])

print('Fetching DoB scaffold/shed filings (citywide)...')
filings = fetch_socrata(
    DOB_FILINGS_URL,
    where="(scaffold='1' OR shed='1') AND latitude IS NOT NULL",
    select=FILINGS_COLS,
    cache_key='dob_scaffold_shed_filings_v1'
)

for col in ['latitude', 'longitude']:
    filings[col] = pd.to_numeric(filings[col], errors='coerce')
for col in ['filing_date', 'first_permit_date', 'current_status_date', 'signoff_date']:
    filings[col] = pd.to_datetime(filings[col], errors='coerce', utc=True)

filings = filings.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
print(f'Filings with coordinates: {len(filings):,}')
print(f'By borough: {dict(filings.borough.value_counts())}')

### 3b. Permit Expiration Lookup (Dataset 2 via BIN)

In [ ]:
# First-pass spatial match to identify which BINs we need
hotspots_gdf = gpd.GeoDataFrame(
    hotspots_df,
    geometry=gpd.points_from_xy(hotspots_df.lon, hotspots_df.lat),
    crs=WGS84
)
hotspots_sp = hotspots_gdf.to_crs(NYC_SP)

filings_gdf = gpd.GeoDataFrame(
    filings,
    geometry=gpd.points_from_xy(filings.longitude, filings.latitude),
    crs=WGS84
)
filings_sp = filings_gdf.to_crs(NYC_SP)

# KD-tree
hs_coords = np.column_stack([hotspots_sp.geometry.x.values, hotspots_sp.geometry.y.values])
fil_coords = np.column_stack([filings_sp.geometry.x.values, filings_sp.geometry.y.values])
tree = cKDTree(fil_coords)
radius_ft = MATCH_RADIUS_M * 3.28084
dists, idxs = tree.query(hs_coords, k=1)

matched_mask = dists <= radius_ft
matched_bins = filings.iloc[idxs[matched_mask]]['bin'].dropna().unique()
print(f'Hotspots within {MATCH_RADIUS_M}m of a filing: {matched_mask.sum():,}')
print(f'Unique BINs to look up: {len(matched_bins):,}')

In [ ]:
# Batch-fetch permit issuance by BIN
PERMITS_COLS = ','.join([
    'job__', 'bin__', 'permit_type', 'permit_subtype', 'permit_status',
    'issuance_date', 'expiration_date'
])

cache_path = CACHE_DIR / 'dob_permits_by_bin_v1.parquet'
if cache_path.exists():
    permits_df = pd.read_parquet(cache_path)
    print(f'Loaded {len(permits_df):,} permit records from cache')
else:
    BATCH_SIZE = 40
    all_permit_records = []
    n_batches = (len(matched_bins) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(matched_bins), BATCH_SIZE):
        batch = matched_bins[i:i + BATCH_SIZE]
        or_clauses = ' OR '.join(f"bin__='{b}'" for b in batch)
        where = f'({or_clauses}) AND issuance_date IS NOT NULL'
        params = {
            '$where': where,
            '$select': PERMITS_COLS,
            '$limit': 50000,
            '$order': ':id',
        }
        r = requests.get(DOB_PERMITS_URL, params=params, timeout=120)
        r.raise_for_status()
        all_permit_records.extend(r.json())
        batch_num = i // BATCH_SIZE + 1
        print(f'  Batch {batch_num}/{n_batches}: {len(all_permit_records):,} permits', end='\r')
    permits_df = pd.DataFrame(all_permit_records)
    if len(permits_df) > 0:
        permits_df.to_parquet(cache_path, index=False)
    print(f'\nFetched {len(permits_df):,} permits for {len(matched_bins):,} BINs')

# Process: per BIN, take latest shed-related permit
if len(permits_df) > 0:
    for col in ['issuance_date', 'expiration_date']:
        permits_df[col] = pd.to_datetime(permits_df[col], errors='coerce', utc=True)
    shed_subtypes = {'SH', 'SD', 'SF'}
    shed_permits = permits_df[permits_df['permit_subtype'].isin(shed_subtypes)]
    src_df = shed_permits if len(shed_permits) > 0 else permits_df
    permits_latest = (
        src_df.sort_values('issuance_date', ascending=False)
        .drop_duplicates(subset='bin__', keep='first')
        [['bin__', 'issuance_date', 'expiration_date', 'permit_status', 'permit_subtype']]
    )
    print(f'Unique BINs with permit data: {len(permits_latest):,}')
else:
    permits_latest = pd.DataFrame(columns=['bin__', 'issuance_date', 'expiration_date',
                                           'permit_status', 'permit_subtype'])

In [ ]:
# Merge filings with permit expiration via BIN
filings_enriched = filings.merge(
    permits_latest, left_on='bin', right_on='bin__', how='left'
).reset_index(drop=True)

def derive_permit_status(row):
    if pd.notna(row.get('expiration_date')):
        return 'active' if row['expiration_date'] > TODAY else 'expired'
    if pd.notna(row.get('signoff_date')):
        return 'completed'
    if pd.notna(row.get('first_permit_date')):
        return 'active_no_expiry_data'
    return 'no_permit'

filings_enriched['permit_lifecycle'] = filings_enriched.apply(derive_permit_status, axis=1)
print('Permit lifecycle (all filings):')
print(filings_enriched.permit_lifecycle.value_counts().to_string())

filings_enriched_sp = gpd.GeoDataFrame(
    filings_enriched,
    geometry=gpd.points_from_xy(filings_enriched.longitude, filings_enriched.latitude),
    crs=WGS84
).to_crs(NYC_SP)

## 4. Spatial Cross-Reference & Classification

For each raster hotspot (estimated scaffold position), find the nearest DoB
scaffold/shed filing within 100m and classify.

In [ ]:
# Rebuild KD-tree with enriched filings
fil_e_coords = np.column_stack([
    filings_enriched_sp.geometry.x.values,
    filings_enriched_sp.geometry.y.values
])
tree_e = cKDTree(fil_e_coords)
dists_e, idxs_e = tree_e.query(hs_coords, k=1)

hotspots_df = hotspots_df.copy()
hotspots_df['nearest_dist_m'] = dists_e / 3.28084
hotspots_df['nearest_idx'] = idxs_e
hotspots_df['has_nearby_filing'] = dists_e <= radius_ft

# Pull matched filing columns
match_cols = [
    'job_filing_number', 'filing_date', 'first_permit_date',
    'signoff_date', 'expiration_date', 'permit_lifecycle',
    'scaffold', 'shed', 'borough', 'house_no', 'street_name',
]
for col in match_cols:
    vals = filings_enriched.iloc[idxs_e][col].values
    hotspots_df[col] = np.where(hotspots_df.has_nearby_filing, vals, None)

for col in ['filing_date', 'first_permit_date', 'signoff_date', 'expiration_date']:
    hotspots_df[col] = pd.to_datetime(hotspots_df[col], errors='coerce', utc=True)

# Classify
def classify(row):
    if not row['has_nearby_filing']:
        return 'unpermitted'
    lc = row['permit_lifecycle']
    if lc in ('active', 'active_no_expiry_data'):
        return 'permitted'
    if lc in ('expired', 'completed'):
        return 'expired'
    return 'unpermitted'

hotspots_df['compliance'] = hotspots_df.apply(classify, axis=1)

print('=' * 60)
print('SCAFFOLDING PERMIT COMPLIANCE (RASTER HOTSPOTS)')
print('=' * 60)
for status in ['permitted', 'expired', 'unpermitted']:
    n = (hotspots_df.compliance == status).sum()
    pct = 100 * n / len(hotspots_df)
    print(f'  {status:12s}: {n:5d}  ({pct:5.1f}%)')
print(f'  {"total":12s}: {len(hotspots_df):5d}')
print(f'\nMatched: {hotspots_df.has_nearby_filing.sum():,}, '
      f'Unmatched: {(~hotspots_df.has_nearby_filing).sum():,}')

## 5. Summary Statistics

In [ ]:
COLORS = {'permitted': '#2ecc71', 'expired': '#f39c12', 'unpermitted': '#e74c3c'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Compliance bar chart
ax = axes[0, 0]
counts = hotspots_df.compliance.value_counts().reindex(['permitted', 'expired', 'unpermitted'])
bars = ax.bar(counts.index, counts.values,
              color=[COLORS[c] for c in counts.index], edgecolor='white', width=0.6)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, count + 5,
            f'{count}\n({100*count/len(hotspots_df):.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Hotspot Count')
ax.set_title('Compliance Status')
ax.set_ylim(0, counts.max() * 1.25)

# (b) Distance to nearest filing
ax = axes[0, 1]
ax.hist(hotspots_df.nearest_dist_m, bins=60, color='#3498db',
        edgecolor='white', alpha=0.8)
ax.axvline(MATCH_RADIUS_M, color='red', ls='--', lw=2,
           label=f'{MATCH_RADIUS_M}m threshold')
ax.set_xlabel('Distance to Nearest Filing (m)')
ax.set_ylabel('Count')
ax.set_title('Hotspot \u2192 Nearest DoB Filing')
ax.legend()

# (c) Max relevance by compliance
ax = axes[1, 0]
for status in ['permitted', 'expired', 'unpermitted']:
    subset = hotspots_df[hotspots_df.compliance == status]
    if len(subset):
        ax.hist(subset.max_value, bins=30, alpha=0.55,
                label=f'{status} (n={len(subset)})', color=COLORS[status])
ax.set_xlabel('Max Raster Relevance')
ax.set_ylabel('Count')
ax.set_title('Detection Confidence by Status')
ax.legend(fontsize=9)

# (d) Scaffold type by compliance
ax = axes[1, 1]
type_compliance = hotspots_df.groupby(['scaffold_type', 'compliance']).size().unstack(fill_value=0)
type_compliance = type_compliance.reindex(columns=['permitted', 'expired', 'unpermitted'], fill_value=0)
type_compliance.plot(kind='bar', ax=ax, color=[COLORS[c] for c in type_compliance.columns],
                     edgecolor='white', width=0.7)
ax.set_xlabel('Scaffold Type')
ax.set_ylabel('Count')
ax.set_title('Scaffold Type \u00d7 Compliance')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'compliance_raster_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed statistics
print('Detailed Statistics by Compliance Status')
print('-' * 65)
for status in ['permitted', 'expired', 'unpermitted']:
    s = hotspots_df[hotspots_df.compliance == status]
    print(f'\n{status.upper()} (n={len(s)}):')
    print(f'  Max relevance: {s.max_value.mean():.3f} \u00b1 {s.max_value.std():.3f}')
    print(f'  Hotspot area:  {s.area_m2.mean():.0f} \u00b1 {s.area_m2.std():.0f} m\u00b2')
    print(f'  Type: {dict(s.scaffold_type.value_counts())}')
    if s.has_nearby_filing.any():
        m = s[s.has_nearby_filing]
        print(f'  Permit dist: {m.nearest_dist_m.mean():.1f} \u00b1 {m.nearest_dist_m.std():.1f} m')

# Comparison with rerank-based notebook
print('\n' + '=' * 65)
print('COMPARISON: Raster vs. Rerank Approach')
print('=' * 65)
print(f'Raster hotspots:     {len(hotspots_df):,} scaffold locations')
print(f'  \u2192 positions represent estimated scaffold location (ray convergence)')
print(f'  \u2192 each aggregates evidence from multiple camera viewpoints')
print(f'  \u2192 includes scaffold type (green/white) classification')

## 6. Interactive Compliance Map

- **Raster overlay** (toggle): green scaffolding relevance heatmap
- **Hotspot markers**: colored by compliance, sized by area, outlined by scaffold type
- **DoB filing heatmap** (toggle): permit density

In [ ]:
# Prepare raster overlay as RGBA image for folium
def raster_to_rgba(data, cmap_name='inferno', vmin=0, vmax=0.5, alpha=180):
    """Convert float32 raster to RGBA uint8 array for folium ImageOverlay."""
    normed = np.clip((data - vmin) / (vmax - vmin), 0, 1)
    colormap = cm.get_cmap(cmap_name)
    rgba = (colormap(normed) * 255).astype(np.uint8)
    # Transparent where no data
    rgba[np.isnan(data), 3] = 0
    # Set alpha for valid pixels
    rgba[~np.isnan(data), 3] = alpha
    return rgba

green_rgba = raster_to_rgba(green, 'Greens', vmin=0, vmax=0.4, alpha=160)
print(f'Raster RGBA shape: {green_rgba.shape}')
print(f'Bounds: S={bounds.bottom}, N={bounds.top}, W={bounds.left}, E={bounds.right}')

In [ ]:
# Build folium map
m = folium.Map(location=[40.78, -73.96], zoom_start=12, tiles='CartoDB positron')

# Layer: green scaffolding raster overlay
ImageOverlay(
    image=green_rgba,
    bounds=[[bounds.bottom, bounds.left], [bounds.top, bounds.right]],
    opacity=0.7,
    name='Green Scaffolding Raster',
    show=False,
).add_to(m)

# Layer: DoB filing density
heat_data = filings[['latitude', 'longitude']].dropna().values.tolist()
HeatMap(heat_data, name='DoB Filing Density', min_opacity=0.25,
        radius=12, blur=15, show=False).add_to(m)

# Layers: hotspot markers by compliance
TYPE_OUTLINE = {'green': '#238b45', 'white': '#7b2d8e', 'ambiguous': '#7f8c8d'}

for status in ['unpermitted', 'expired', 'permitted']:
    color = COLORS[status]
    n = (hotspots_df.compliance == status).sum()
    group = folium.FeatureGroup(name=f'Hotspots \u2014 {status.title()} ({n})')
    subset = hotspots_df[hotspots_df.compliance == status]

    for _, row in subset.iterrows():
        outline = TYPE_OUTLINE.get(row.scaffold_type, '#7f8c8d')
        radius = max(5, min(12, row.n_pixels))

        popup_parts = [
            f"<b style='color:{color}'>{status.upper()}</b> "
            f"<span style='color:{outline}'>({row.scaffold_type})</span>",
            f"<b>Relevance:</b> {row.max_value:.3f} (max), {row.mean_value:.3f} (mean)",
            f"<b>Area:</b> {row.area_m2:.0f} m\u00b2 ({row.n_pixels} pixels)",
            f"<b>Location:</b> ({row.lat:.6f}, {row.lon:.6f})",
        ]
        if row.has_nearby_filing:
            popup_parts += [
                '<hr style="margin:4px 0">',
                f"<b>Nearest filing:</b> {row.job_filing_number}",
                f"<b>Distance:</b> {row.nearest_dist_m:.0f}m",
                f"<b>Address:</b> {row.house_no} {row.street_name}",
                f"<b>Filed:</b> {str(row.filing_date)[:10] if pd.notna(row.filing_date) else 'N/A'}",
                f"<b>Permit issued:</b> {str(row.first_permit_date)[:10] if pd.notna(row.first_permit_date) else 'N/A'}",
                f"<b>Expires:</b> {str(row.expiration_date)[:10] if pd.notna(row.expiration_date) else 'N/A'}",
                f"<b>Signed off:</b> {str(row.signoff_date)[:10] if pd.notna(row.signoff_date) else 'N/A'}",
            ]

        folium.CircleMarker(
            location=[row.lat, row.lon],
            radius=radius,
            color=outline,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup('<br>'.join(popup_parts), max_width=350),
            tooltip=f'{status} | {row.scaffold_type} | rel={row.max_value:.2f}',
        ).add_to(group)

    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:2px solid #bbb; font-size:13px; line-height:1.7;
     box-shadow:0 2px 6px rgba(0,0,0,0.15);">
<b>Scaffolding Compliance</b><br>
<span style="background:#2ecc71;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Permitted<br>
<span style="background:#f39c12;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Expired / completed<br>
<span style="background:#e74c3c;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Unpermitted<br>
<hr style="margin:4px 0">
<b>Outline = scaffold type</b><br>
<span style="border:2px solid #238b45;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Green (standard)<br>
<span style="border:2px solid #7b2d8e;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> White / arched<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

map_path = OUTPUT_DIR / 'scaffolding_compliance_raster.html'
m.save(str(map_path))
print(f'Map saved: {map_path}')
print(f'Hotspots: {len(hotspots_df):,}')
m

## 7. Export Results

In [ ]:
export_cols = [
    'lat', 'lon', 'n_pixels', 'area_m2',
    'max_value', 'mean_value', 'signed_diff', 'scaffold_type',
    'compliance', 'nearest_dist_m', 'has_nearby_filing',
    'job_filing_number', 'filing_date', 'first_permit_date',
    'expiration_date', 'signoff_date', 'permit_lifecycle',
    'scaffold', 'shed', 'house_no', 'street_name',
]
export_cols = [c for c in export_cols if c in hotspots_df.columns]
export_df = hotspots_df[export_cols].copy()

export_path = OUTPUT_DIR / 'scaffolding_compliance_raster_classified.parquet'
export_df.to_parquet(export_path, index=False)

print(f'Exported {len(export_df):,} classified hotspots to:')
print(f'  {export_path}')
print(f'\nCompliance:')
print(export_df.compliance.value_counts().to_string())
print(f'\nScaffold type:')
print(export_df.scaffold_type.value_counts().to_string())

In [ ]:
# Cleanup rasterio handles
for s in [src_green, src_white, src_signed, src_abs]:
    s.close()